# Cohort statistics (mcPHASES)

Variable definitions and file semantics follow `data/README.txt` (e.g. `id`, `study_interval` as Interval 1 vs 2, `subject-info.csv` demographics, `height_and_weight.csv` in cm and kg).

This notebook builds a **Table 1–style** summary: participant counts by collection interval, age at the calendar year of that interval, anthropometrics when present, and categorical demographics.

**Dependency:** `pandas` (install in your environment, e.g. `pip install pandas`).

In [ ]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"

pd.options.display.max_colwidth = 120
pd.options.display.width = 140

In [ ]:
def load_subject_info(path: Path) -> pd.DataFrame:
    """Baseline survey fields described in README (subject-info.csv)."""
    return pd.read_csv(path)


def load_height_weight(path: Path) -> pd.DataFrame:
    """Self-reported height (cm) and weight (kg) for 2022 vs 2024 waves."""
    return pd.read_csv(path)


def participant_intervals_from_modal_file(path: Path) -> pd.DataFrame:
    """
    Distinct (id, study_interval) pairs from a longitudinal table that includes
    README keys `id` and `study_interval` (here: active_minutes.csv).
    """
    usecols = ["id", "study_interval"]
    df = pd.read_csv(path, usecols=usecols)
    return df.drop_duplicates()


def interval_calendar_year(study_interval):
    """Map exported study_interval label to approximate calendar year for age."""
    if pd.isna(study_interval):
        return pd.NA
    return int(study_interval)


def mean_sd(x: pd.Series) -> str:
    x = pd.to_numeric(x, errors="coerce").dropna()
    if x.empty:
        return "—"
    return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"


def n_pct(n: int, denom: int) -> str:
    if denom == 0:
        return "—"
    return f"{n} ({100 * n / denom:.1f}%)"


def add_bmi(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    h22, w22 = pd.to_numeric(out["height_2022"], errors="coerce"), pd.to_numeric(
        out["weight_2022"], errors="coerce"
    )
    h24, w24 = pd.to_numeric(out["height_2024"], errors="coerce"), pd.to_numeric(
        out["weight_2024"], errors="coerce"
    )
    out["bmi_2022"] = w22 / ((h22 / 100.0) ** 2)
    out["bmi_2024"] = w24 / ((h24 / 100.0) ** 2)
    return out

In [ ]:
subject = load_subject_info(DATA_DIR / "subject-info.csv")
hw = add_bmi(load_height_weight(DATA_DIR / "height_and_weight.csv"))
interval_pairs = participant_intervals_from_modal_file(DATA_DIR / "active_minutes.csv")

subject["id"] = subject["id"].astype(int)
hw["id"] = hw["id"].astype(int)
interval_pairs["id"] = interval_pairs["id"].astype(int)

cohort = interval_pairs.merge(subject, on="id", how="left").merge(hw, on="id", how="left")

cohort["study_interval"] = pd.to_numeric(cohort["study_interval"], errors="coerce").astype("Int64")
cohort["calendar_year"] = cohort["study_interval"].map(interval_calendar_year)
cohort["age_years"] = cohort["calendar_year"] - pd.to_numeric(cohort["birth_year"], errors="coerce")

mask_2022 = cohort["study_interval"] == 2022
cohort["height_cm"] = pd.to_numeric(
    cohort["height_2022"].where(mask_2022, cohort["height_2024"]), errors="coerce"
)
cohort["weight_kg"] = pd.to_numeric(
    cohort["weight_2022"].where(mask_2022, cohort["weight_2024"]), errors="coerce"
)
cohort["bmi"] = pd.to_numeric(
    cohort["bmi_2022"].where(mask_2022, cohort["bmi_2024"]), errors="coerce"
)

cohort.head()

## Participant counts

Rows are distinct **participant × study interval** assignments inferred from device data (`active_minutes.csv` shares the README join keys `id`, `study_interval`).

In [ ]:
n_by_interval = cohort.groupby("study_interval", as_index=False).size().rename(columns={"size": "n_participants"})
n_total = cohort["id"].nunique()
n_both = cohort.groupby("id")["study_interval"].nunique().eq(2).sum()

counts = pd.DataFrame(
    [
        {"stratum": "All unique participants", "n": n_total},
        {"stratum": "Participants in both intervals (same id)", "n": int(n_both)},
    ]
)
display(counts)
display(n_by_interval)

## Continuous variables: mean (SD) by `study_interval`

- **Age**: `calendar_year - birth_year` from `subject-info.csv` (`birth_year`).
- **Height / weight / BMI**: wave-matched columns from `height_and_weight.csv` (`height_2022`/`weight_2022` vs `height_2024`/`weight_2024`). BMI = kg / (m)².

In [ ]:
continuous = ["age_years", "height_cm", "weight_kg", "bmi"]
cont_rows = []
for var, label in [
    ("age_years", "Age (years)"),
    ("height_cm", "Height (cm)"),
    ("weight_kg", "Weight (kg)"),
    ("bmi", "BMI (kg/m²)"),
]:
    row = {"variable": label}
    for interval in sorted(cohort["study_interval"].dropna().unique()):
        sub = cohort.loc[cohort["study_interval"] == interval, var]
        row[f"{interval} (n={sub.notna().sum()})"] = mean_sd(sub)
    row["All intervals"] = mean_sd(cohort[var])
    cont_rows.append(row)

continuous_table = pd.DataFrame(cont_rows)
continuous_table

## Categorical variables: n (%) within each `study_interval`

Columns follow `subject-info.csv` as documented in README (`gender`, `ethnicity`, `education`, `sexually_active`, `self_report_menstrual_health_literacy`). Percent denominator is non-missing responses for that variable in the stratum.

In [ ]:
cat_specs = [
    ("gender", "Gender"),
    ("ethnicity", "Ethnicity (self-report)"),
    ("education", "Education"),
    ("sexually_active", "Sexually active"),
    ("self_report_menstrual_health_literacy", "Menstrual health literacy (self-report)"),
]

cat_parts = []
for col, title in cat_specs:
    if col not in cohort.columns:
        continue
    for interval in sorted(cohort["study_interval"].dropna().unique()):
        sub = cohort.loc[cohort["study_interval"] == interval, col].astype("string")
        denom = sub.notna() & sub.str.strip().ne("")
        d = int(denom.sum())
        vc = sub[denom].value_counts(dropna=False)
        for level, c in vc.items():
            cat_parts.append(
                {
                    "domain": title,
                    "level": str(level),
                    "study_interval": interval,
                    "value": n_pct(int(c), d),
                }
            )

cat_long = pd.DataFrame(cat_parts)
if not cat_long.empty:
    cat_table = cat_long.pivot_table(
        index=["domain", "level"], columns="study_interval", values="value", aggfunc="first"
    ).reset_index()
else:
    cat_table = pd.DataFrame()

cat_table

## Age at menarche (continuous)

From README / `subject-info.csv`: `age_of_first_menarche` (years).

In [ ]:
menarche_rows = []
for interval in sorted(cohort["study_interval"].dropna().unique()):
    sub = cohort.loc[cohort["study_interval"] == interval, "age_of_first_menarche"]
    menarche_rows.append(
        {
            "study_interval": interval,
            "age_of_first_menarche_mean_sd": mean_sd(sub),
            "n_nonmissing_menarche": int(pd.to_numeric(sub, errors="coerce").notna().sum()),
        }
    )
menarche_table = pd.DataFrame(menarche_rows)
menarche_table

## Combined output

In [ ]:
from IPython.display import Markdown, display

display(Markdown("# Cohort statistics (combined)"))
display(Markdown("## Participant counts"))
display(Markdown("**Overall**"))
display(counts)
display(Markdown("**By study interval**"))
display(n_by_interval)

display(Markdown("## Continuous variables: mean (SD)"))
display(continuous_table)

display(Markdown("## Categorical variables: n (%)"))
if not cat_table.empty:
    display(cat_table)
else:
    display(Markdown("_No categorical breakdown (empty or columns missing)._"))

display(Markdown("## Age at first menarche"))
display(menarche_table)

## Combined export (optional)

Writes CSVs next to this notebook for use in manuscripts or slides.

In [ ]:
out_dir = Path(".").resolve() / "cohort_table_outputs"
out_dir.mkdir(exist_ok=True)

n_by_interval.to_csv(out_dir / "participant_counts_by_interval.csv", index=False)
counts.to_csv(out_dir / "participant_counts_overall.csv", index=False)
continuous_table.to_csv(out_dir / "continuous_mean_sd.csv", index=False)
if not cat_table.empty:
    cat_table.to_csv(out_dir / "categorical_n_pct.csv", index=False)
menarche_table.to_csv(out_dir / "menarche_by_interval.csv", index=False)

print("Wrote:", out_dir)